## F1 Pit Stop Prediction - Playground Series S6E5

### Objective
Predict whether an F1 driver will pit on the next lap (binary classification).

### Dataset
- Train: 439,140 rows, 16 columns
- Test: 188,165 rows, 15 columns  
- Target: PitNextLap (19.9% positive class)

### Approach
- Model: LightGBM binary classifier
- Metric: ROC AUC
- Validation: 80/20 train-test split with stratification

### Step-1: Importing libraries

In [4]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

### Step-2: Data Loading

In [5]:
# Load data
print("Loading data...")
train = pd.read_csv('train.csv')  # Kaggle path
test = pd.read_csv('test.csv')
sample_sub = pd.read_csv('sample_submission.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(f"Target distribution:\n{train['PitNextLap'].value_counts(normalize=True)}")

Loading data...
Train shape: (439140, 16)
Test shape: (188165, 15)
Target distribution:
PitNextLap
0.0    0.801018
1.0    0.198982
Name: proportion, dtype: float64


### Step-3: Feature Preparation

In [6]:
# Prepare features
print("\nPreparing features...")
target = 'PitNextLap'
id_col = 'id'

# Get feature columns
feature_cols = [col for col in train.columns if col not in [target, id_col]]

X = train[feature_cols].copy()  # Use .copy() to avoid warning
y = train[target].copy()
X_test = test[feature_cols].copy()

# Handle categorical columns (if they exist)
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
if cat_cols:
    print(f"Encoding categorical columns: {cat_cols}")
    for col in cat_cols:
        X[col] = pd.Categorical(X[col]).codes
        X_test[col] = pd.Categorical(X_test[col]).codes


Preparing features...
Encoding categorical columns: ['Driver', 'Compound', 'Race']


### Step-4: Train-Test Split

In [8]:
# Train/validation split
print("\nSplitting data...")
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("\nData split completed!")


Splitting data...

Data split completed!


### Step-5: Model Training (LightGBM)

In [9]:
# Train LightGBM model
print("\nTraining model...")
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
    'verbosity': -1,
    'seed': 42
}

train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

model = lgb.train(
    params,
    train_data,
    num_boost_round=500,
    valid_sets=[val_data],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

# Validate
val_preds = model.predict(X_val, num_iteration=model.best_iteration)
val_auc = roc_auc_score(y_val, val_preds)
print(f'\n✓ Validation AUC: {val_auc:.5f}')


Training model...
Training until validation scores don't improve for 50 rounds
[100]	valid_0's auc: 0.938403
[200]	valid_0's auc: 0.943188
[300]	valid_0's auc: 0.944846
[400]	valid_0's auc: 0.945872
[500]	valid_0's auc: 0.946642
Did not meet early stopping. Best iteration is:
[500]	valid_0's auc: 0.946642

✓ Validation AUC: 0.94664


### Step-6: Generating Submission

In [10]:
# Generate submission
print("\nGenerating predictions...")
test_preds = model.predict(X_test, num_iteration=model.best_iteration)

submission = pd.DataFrame({
    'id': test[id_col],
    'PitNextLap': test_preds
})
\
# Save
submission.to_csv('submission.csv', index=False)
print("\n✓ Submission file created!")
print(f"Prediction range: [{test_preds.min():.4f}, {test_preds.max():.4f}]")
print(f"Mean prediction: {test_preds.mean():.4f}")
print("\nFirst 5 rows:")
print(submission.head())


Generating predictions...

✓ Submission file created!
Prediction range: [0.0002, 0.9822]
Mean prediction: 0.1984

First 5 rows:
       id  PitNextLap
0  439140    0.003751
1  439141    0.003655
2  439142    0.003130
3  439143    0.166250
4  439144    0.826954


### Step-7: View Output CSV

In [11]:
from IPython.display import display

print("Submission file (first 10 rows):")
display(submission.head(10))

print(f"\nSubmission stats:")
print(f"- Total rows: {len(submission)}")
print(f"- Prediction range: [{submission['PitNextLap'].min():.4f}, {submission['PitNextLap'].max():.4f}]")
print(f"- Mean prediction: {submission['PitNextLap'].mean():.4f}")

Submission file (first 10 rows):


,id,PitNextLap
0,439140,0.003751
1,439141,0.003655
2,439142,0.003130
3,439143,0.166250
4,439144,0.826954
5,439145,0.297992
6,439146,0.001988
7,439147,0.027035
8,439148,0.041520
9,439149,0.002270



Submission stats:
- Total rows: 188165
- Prediction range: [0.0002, 0.9822]
- Mean prediction: 0.1984
